# 03 HOMER Run

Do HOMER enrichment of clean full peaks.

In [2]:
cd /home/dalbao/AlbaoRunx3Manuscript/cutnrun

docker_run() {
    docker run --rm -i \
        -u $(id -u):$(id -g) \
        -v /home/dalbao:/home/dalbao \
        -v /etc/timezone:/etc/timezone:ro \
        -v /etc/localtime:/etc/localtime:ro \
        -w $(pwd) \
        --cpus=24 \
        "$@"
}

# Define software to use:
## homer for motif enrichment analysis
homer() {
    docker_run dsalbao/homer:5.1 "$@"
}

# Check clean full peaks in 01_peakEDA
ls -lah 01_peakEDA

total 432K
drwxr-xr-x 7 1001 1000 4.0K Jul 21 23:44 .
drwxr-xr-x 4 1001 1000 4.0K Jul 22 00:05 ..
drwxr-xr-x 4 1001 1000 4.0K Jul 21 22:58 02_initialHeatmaps
drwxr-xr-x 6 1001 1000 4.0K Jul 21 23:13 03_peakCleanup
drwxr-xr-x 4 1001 1000 4.0K Jul 21 23:49 04_peakConsolidate
-rw-r--r-- 1 1001 1000  57K Jul 21 23:46 cluster1.fullpeak.clean.bed
-rw-r--r-- 1 1001 1000 143K Jul 21 23:45 cluster2.fullpeak.clean.bed
-rw-r--r-- 1 1001 1000 200K Jul 21 23:42 fullpeak.clean.clustered.named.bed
drwxr-xr-x 2 1001 1000 4.0K Jul 21 22:51 max_signal
drwxr-xr-x 2 1001 1000 4.0K Jul 21 22:51 named_peaks


In [3]:
# Export to run using GNU parallel
export -f docker_run homer
export SHELL=/bin/bash
export NCPU=24

GENOME=source_data/260713_reBAM2_DeDup_CPM/04_reporting/igv/genome.fa

In [4]:
mkdir -p 03_motif

# Run two HOMER jobs in parallel, one for each cluster of peaks
parallel -j2 --joblog 03_motif/homer.joblog --resume-failed \
    homer findMotifsGenome.pl \
        01_peakEDA/cluster{}.fullpeak.clean.bed "$GENOME" \
        03_motif/cluster{} \
        -size given -p 24 \
    ::: 1 2


	Position file = 01_peakEDA/cluster2.fullpeak.clean.bed
	Genome = source_data/260713_reBAM2_DeDup_CPM/04_reporting/igv/genome.fa
	Output Directory = 03_motif/cluster2
	Using actual sizes of regions (-size given)
	Fragment size set to given
	Using 24 CPUs
	Using Custom Genome
	Peak/BED file conversion summary:
		BED/Header formatted lines: 2982
		peakfile formatted lines: 0

	Peak File Statistics:
		Total Peaks: 2982
		Redundant Peak IDs: 0
		Peaks lacking information: 0 (need at least 5 columns per peak)
		Peaks with misformatted coordinates: 0 (should be integer)
		Peaks with misformatted strand: 0 (should be either +/- or 0/1)

	Peak file looks good!

	Background fragment size set to 1096 (avg size of targets)
	Could not find background files for 1096 bp fragments
	Below are the sizes that are already available prepared.
		200
		1006
		1072
	HOMER will now create background files for 1096 bp fragments
	To CANCEL and rerun with a differet "-size <#>", hit <CTRL+C> now!
		5
		4
		3
		

In [5]:
# Convert homer results to a more readable format (e.g., CSV) for downstream analysis or visualization.
perl convertHomerTsv.pl 03_motif/cluster1/knownResults.txt 03_motif/cluster1.csv
perl convertHomerTsv.pl 03_motif/cluster2/knownResults.txt 03_motif/cluster2.csv

### Motif Extraction

Extract motifs of interest for analysis

In [7]:
cd /home/dalbao/AlbaoRunx3Manuscript/cutnrun

# Define Runx3 motifs
runx_motifs=(1 2 3 4)
composite_motifs=(9)
tbox_motifs=(65 111)
ets_motifs=(5 6 7)
bzip_motifs=(20 23 24 25 26 27 28 29 35)
klf_motifs=(22 30 32 33)
tcfstat_motifs=(277 311 285 300 310)
eomes_motifs=(316)

# Directory containing known*.motif files
motif_dir="03_motif/cluster1/knownResults"

# Output directory for combined motif files
out_dir="03_motif/motif_files"
mkdir -p "$out_dir"

# Function to combine motif files given a name and an array of numbers
combine_motifs () {
    local name="$1"
    shift
    local numbers=("$@")
    local out_file="${out_dir}/${name}.motif"

    > "$out_file"  # truncate/create output file
    for n in "${numbers[@]}"; do
        motif_file="${motif_dir}/known${n}.motif"
        if [[ -f "$motif_file" ]]; then
            cat "$motif_file" >> "$out_file"
        else
            echo "Warning: $motif_file not found" >&2
        fi
    done
    echo "Created $out_file"
}

# Combine motifs into named files
combine_motifs "Runx" "${runx_motifs[@]}"
combine_motifs "Composite" "${composite_motifs[@]}"
combine_motifs "Tbox" "${tbox_motifs[@]}"
combine_motifs "ETS" "${ets_motifs[@]}"
combine_motifs "bZIP" "${bzip_motifs[@]}"
combine_motifs "KLF" "${klf_motifs[@]}"

# Directory containing known*.motif files
motif_dir="03_motif/cluster2/knownResults"
combine_motifs "Tcf.Stat" "${tcfstat_motifs[@]}"
combine_motifs "Eomes" "${eomes_motifs[@]}"

# Combine all motif files into a single file
cat "${out_dir}"/*.motif > "${out_dir}/all.motifs"

Created 03_motif/motif_files/Runx.motif
Created 03_motif/motif_files/Composite.motif
Created 03_motif/motif_files/Tbox.motif
Created 03_motif/motif_files/ETS.motif
Created 03_motif/motif_files/bZIP.motif
Created 03_motif/motif_files/KLF.motif
Created 03_motif/motif_files/Tcf.Stat.motif
Created 03_motif/motif_files/Eomes.motif


### Annotate Motifs

In [8]:
homer findMotifsGenome.pl \
    01_peakEDA/fullpeak.clean.clustered.named.bed \
    $GENOME \
    03_motif/parameters \
    -find 03_motif/motif_files/all.motifs > "03_motif/motifAnnotated.peaks"


	Position file = 01_peakEDA/fullpeak.clean.clustered.named.bed
	Genome = source_data/260713_reBAM2_DeDup_CPM/04_reporting/igv/genome.fa
	Output Directory = 03_motif/parameters
	Will find motif(s) in 03_motif/motif_files/all.motifs
	Using Custom Genome
	Peak/BED file conversion summary:
		BED/Header formatted lines: 4172
		peakfile formatted lines: 0

	Peak File Statistics:
		Total Peaks: 4172
		Redundant Peak IDs: 0
		Peaks lacking information: 0 (need at least 5 columns per peak)
		Peaks with misformatted coordinates: 0 (should be integer)
		Peaks with misformatted strand: 0 (should be either +/- or 0/1)

	Peak file looks good!

	Background files for 200 bp fragments found.
	Custom genome sequence file: source_data/260713_reBAM2_DeDup_CPM/04_reporting/igv/genome.fa

	Extracting sequences from file: source_data/260713_reBAM2_DeDup_CPM/04_reporting/igv/genome.fa
	Looking for peak sequences in a single file (source_data/260713_reBAM2_DeDup_CPM/04_reporting/igv/genome.fa)
	Extracting 245

In [11]:
# Simplify
perl simplify_motif_tsv.pl 03_motif/motifAnnotated.peaks > 03_motif/motifAnnotated.simple.tsv